# 🔬 AMD Detection Tool - Python/Google Colab Edition

**Version:** 1.5.1  
**Based on:** Rockwell et al. (2021) USGS methodology

This notebook detects **Acid Mine Drainage (AMD)** and **iron sulfate minerals** using satellite imagery from Landsat 8 and Sentinel-2.

---

## 📋 Table of Contents

1. [Setup & Installation](#1-setup--installation)
2. [Google Earth Engine Authentication](#2-google-earth-engine-authentication)
3. [Define Study Area](#3-define-study-area)
4. [Run AMD Analysis](#4-run-amd-analysis)
5. [Visualize Results](#5-visualize-results)
6. [Export Results](#6-export-results)
7. [Advanced: Custom Analysis](#7-advanced-custom-analysis)

---

## 1. Setup & Installation

First, we install the required packages and import the AMD detection module.

In [ ]:
# ============================================================================
# STEP 1.1: Install Required Packages (Run once)
# ============================================================================

# Uncomment the following lines if running in Google Colab
# !pip install earthengine-api geemap folium --quiet

print("✅ Packages ready!")

In [ ]:
# ============================================================================
# STEP 1.2: Import Libraries + Load AMD Module (Colab-safe)
# ============================================================================

import os
import sys

import ee
import geemap

# --- Load amd_detection module ---
# Why this is needed:
# In Colab/Jupyter, `import amd_detection` works only if amd_detection.py is present
# in the current working directory (or installed as a package).
#
# This cell supports 3 workflows:
# 1) Upload `amd_detection.py` manually into Colab (fastest, always works)
# 2) Download `amd_detection.py` from GitHub (ONLY works if the repo is public)
# 3) Clone the repo (ONLY works if the repo is public, or you provide a token)

MODULE_FILENAME = 'amd_detection.py'
REPO = 'coodawy/AMD-Detection-Tool'
RAW_URLS = [
    # Standard raw URL (public repos)
    f'https://raw.githubusercontent.com/{REPO}/main/python/amd_detection.py',
    # Alternative raw URL format (sometimes used by GitHub)
    f'https://raw.githubusercontent.com/{REPO}/refs/heads/main/python/amd_detection.py',
]


def _ensure_in_syspath(path: str) -> None:
    if path not in sys.path:
        sys.path.insert(0, path)


def _try_download_module() -> bool:
    import urllib.request
    from urllib.error import HTTPError

    for url in RAW_URLS:
        try:
            print(f'⬇️ Downloading amd_detection.py from: {url}')
            urllib.request.urlretrieve(url, MODULE_FILENAME)
            print('✅ Downloaded amd_detection.py')
            return True
        except HTTPError as e:
            if e.code == 404:
                # Common if repo is private or the path is wrong
                print(f'❌ 404 Not Found for: {url}')
            else:
                print(f'❌ HTTP error for: {url} -> {e}')
        except Exception as e:
            print(f'❌ Download failed for: {url} -> {e}')

    return False


def _try_clone_repo() -> bool:
    # This only works if the repo is public OR you provide credentials.
    try:
        print('⬇️ Attempting to clone the repo...')
        # NOTE: In Colab, `!` commands run in a shell.
        # If this fails with authentication, the repo is private.
        get_ipython().system(f'git clone https://github.com/{REPO}.git')

        repo_python_dir = os.path.join(os.getcwd(), 'AMD-Detection-Tool', 'python')
        if os.path.exists(os.path.join(repo_python_dir, MODULE_FILENAME)):
            _ensure_in_syspath(repo_python_dir)
            print('✅ Repo cloned and python path configured')
            return True

        print('❌ Repo cloned but amd_detection.py not found in expected path')
        return False
    except Exception as e:
        print('❌ Clone failed:', e)
        return False


# 1) If user uploaded file, use it
if os.path.exists(MODULE_FILENAME):
    print('✅ Found amd_detection.py in current folder')
    _ensure_in_syspath(os.getcwd())
else:
    # 2) Try downloading from GitHub (public repos only)
    downloaded = _try_download_module()
    if downloaded:
        _ensure_in_syspath(os.getcwd())
    else:
        # 3) Try cloning repo (public repos only)
        cloned = _try_clone_repo()
        if not cloned:
            print('\n⚠️ Could not fetch amd_detection.py automatically.')
            print('Most common cause: the GitHub repo is PRIVATE (raw URLs return 404 in that case).')
            print('\n✅ Fix options:')
            print('1) Upload `amd_detection.py` into Colab (recommended)')
            print('2) Make the repo public, then re-run this cell')
            print('3) If you must keep it private: clone with a GitHub token (PAT)')
            print("   Example (replace TOKEN):")
            print(f"   !git clone https://TOKEN@github.com/{REPO}.git")
            raise FileNotFoundError('amd_detection.py not available in Colab environment')


import amd_detection as amd
amd.print_version()

---

## 2. Google Earth Engine Authentication

You need a Google Earth Engine account. Sign up free at: https://earthengine.google.com/

In [ ]:
# ============================================================================
# STEP 2.1: Authenticate and Initialize GEE
# ============================================================================

# This will open a browser window for authentication
# Follow the prompts and paste the authorization code

amd.initialize_gee()

# If you have a specific GEE project, use:
# amd.initialize_gee(project="your-project-id")

---

## 3. Define Study Area

Choose from pre-defined AMD sites or define your own custom area.

In [ ]:
# ============================================================================
# STEP 3.1: List Available Pre-defined Study Areas
# ============================================================================

print("📍 Available Pre-defined Study Areas:")
print("═" * 50)
for name in amd.STUDY_AREAS.keys():
    print(f"  • {name}")
print("═" * 50)

In [ ]:
# ============================================================================
# STEP 3.2: Select Study Area
# ============================================================================

# OPTION A: Use a pre-defined study area
study_area_name = "Iron Mountain, CA"  # Change this to your preferred site
geometry = amd.get_study_area(study_area_name)

# OPTION B: Define custom coordinates (uncomment to use)
# latitude = 40.6722
# longitude = -122.5278
# buffer_meters = 12000
# geometry = ee.Geometry.Point([longitude, latitude]).buffer(buffer_meters)
# study_area_name = "Custom Area"

print(f"✅ Study area selected: {study_area_name}")

In [ ]:
# ============================================================================
# STEP 3.3: Set Date Range
# ============================================================================

# Summer months are recommended to avoid snow/ice interference
start_date = "2023-06-01"
end_date = "2023-09-30"

print(f"📅 Date range: {start_date} to {end_date}")

In [ ]:
# ============================================================================
# STEP 3.4: Select Satellite Sensor
# ============================================================================

# Options: "Landsat 8" (30m resolution) or "Sentinel-2" (10m resolution)
sensor = "Landsat 8"

# Maximum cloud cover percentage for filtering images
max_cloud_cover = 30

print(f"🛰️ Sensor: {sensor}")
print(f"☁️ Max cloud cover: {max_cloud_cover}%")

---

## 4. Run AMD Analysis

This section runs the complete AMD detection analysis.

In [ ]:
# ============================================================================
# STEP 4.1: Run Complete Analysis
# ============================================================================

print("🔄 Running AMD analysis...")
print(f"   Study Area: {study_area_name}")
print(f"   Date Range: {start_date} to {end_date}")
print(f"   Sensor: {sensor}")
print()

# Run the analysis
results = amd.analyze_region(
    geometry=geometry,
    start_date=start_date,
    end_date=end_date,
    sensor=sensor,
    cloud_cover_max=max_cloud_cover
)

print("✅ Analysis complete!")
print()
print("📊 Results contain:")
print("   • composite: Satellite image with spectral indices")
print("   • land_classification: 19-class AMD mineral map")
print("   • water_quality: Water contamination assessment")
print("   • iron_sulfate: Iron sulfate index layer")

In [ ]:
# ============================================================================
# STEP 4.2: View Classification Legend
# ============================================================================

amd.print_class_legend()

---

## 5. Visualize Results

Create interactive maps to explore the AMD detection results.

In [ ]:
# ============================================================================
# STEP 5.1: Create Interactive Map
# ============================================================================

# Create map centered on study area
Map = amd.create_map(zoom=12)

# Add all result layers
Map = amd.add_results_to_map(Map, results, geometry)

# Add true color composite for reference
true_color_vis = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0, 'max': 0.3}
Map.addLayer(results['composite'], true_color_vis, 'True Color', False)

# Center map on study area
Map.centerObject(geometry, 12)

# Display the map
Map

In [ ]:
# ============================================================================
# STEP 5.2: View Individual Layers (Alternative Visualization)
# ============================================================================

# Create a new map for iron sulfate only
iron_map = amd.create_map(zoom=12)

# Add iron sulfate index with custom visualization
iron_vis = {
    'min': 1.0,
    'max': 2.5,
    'palette': ['yellow', 'orange', 'red', 'darkred']
}
iron_map.addLayer(results['iron_sulfate'], iron_vis, 'Iron Sulfate Index')
iron_map.centerObject(geometry, 12)

print("🔴 Iron Sulfate Index Map:")
print("   Yellow = Low (1.0-1.5)")
print("   Orange = Moderate (1.5-2.0)")
print("   Red = High (2.0-2.5)")
print("   Dark Red = Very High (>2.5)")

iron_map

---

## 6. Export Results

Export classification results to Google Drive for further analysis in GIS software.

In [ ]:
# ============================================================================
# STEP 6.1: Export Land Classification to Google Drive
# ============================================================================

# Create export task
export_task = amd.export_to_drive(
    image=results['land_classification'],
    description=f"AMD_Classification_{study_area_name.replace(' ', '_').replace(',', '')}",
    folder="AMD_Detection_Results",
    region=geometry,
    scale=30  # 30m for Landsat, use 10 for Sentinel-2
)

# Start the export (uncomment to run)
# export_task.start()
# print("✅ Export started! Check your Google Drive in a few minutes.")

print("💡 To start the export, uncomment the lines above and run this cell.")

In [ ]:
# ============================================================================
# STEP 6.2: Export Multiple Layers (Batch Export)
# ============================================================================

# Define exports
exports = [
    ('land_classification', 'AMD_Land_Classification'),
    ('iron_sulfate', 'AMD_Iron_Sulfate_Index'),
]

# Create tasks
tasks = []
for layer_name, export_name in exports:
    task = amd.export_to_drive(
        image=results[layer_name],
        description=export_name,
        folder="AMD_Detection_Results",
        region=geometry,
        scale=30
    )
    tasks.append((export_name, task))
    print(f"📦 Export task created: {export_name}")

print()
print("💡 To start all exports, run:")
print("   for name, task in tasks:")
print("       task.start()")
print("       print(f'Started: {name}')")

---

## 7. Advanced: Custom Analysis

For advanced users who want to customize thresholds or analyze specific indices.

In [ ]:
# ============================================================================
# STEP 7.1: View Default Settings
# ============================================================================

print("⚙️ Default Detection Settings:")
print("═" * 50)
for key, value in amd.DEFAULT_SETTINGS.items():
    print(f"  {key}: {value}")
print("═" * 50)

In [ ]:
# ============================================================================
# STEP 7.2: Run Analysis with Custom Settings
# ============================================================================

# Create custom settings (copy defaults and modify)
custom_settings = amd.DEFAULT_SETTINGS.copy()

# Example: Adjust iron sulfate threshold for more/less sensitive detection
custom_settings["iron_sulfate_threshold"] = 1.10  # Lower = more sensitive
custom_settings["ndvi_max"] = 0.20  # Stricter vegetation mask

# Run with custom settings
custom_results = amd.analyze_region(
    geometry=geometry,
    start_date=start_date,
    end_date=end_date,
    sensor=sensor,
    settings=custom_settings
)

print("✅ Custom analysis complete!")

In [ ]:
# ============================================================================
# STEP 7.3: Compare Results (Default vs Custom)
# ============================================================================

compare_map = amd.create_map(zoom=12)

# Add default results
compare_map.addLayer(
    results['land_classification'],
    {'min': 0, 'max': 19, 'palette': amd.CLASSIFICATION_PALETTE},
    'Default Classification'
)

# Add custom results
compare_map.addLayer(
    custom_results['land_classification'],
    {'min': 0, 'max': 19, 'palette': amd.CLASSIFICATION_PALETTE},
    'Custom Classification (Sensitive)',
    False  # Hidden by default
)

compare_map.centerObject(geometry, 12)
print("💡 Toggle layers in the map controls to compare results.")
compare_map

In [ ]:
# ============================================================================
# STEP 7.4: Enable Adaptive Thresholds (Paper Section 3.3)
# ============================================================================

# Adaptive thresholds are recommended for heterogeneous terrain (mountains)
# They calculate: threshold = mean + (N × std_dev)

adaptive_settings = amd.DEFAULT_SETTINGS.copy()
adaptive_settings["use_adaptive_thresholds"] = True
adaptive_settings["iron_std_mult"] = 2.0  # 2 standard deviations above mean
adaptive_settings["ferric_std_mult"] = 1.5

print("📊 Adaptive thresholds enabled!")
print("   Iron multiplier: 2.0σ")
print("   Ferric multiplier: 1.5σ")
print()
print("💡 Use this for mountainous/heterogeneous terrain.")

---

## 📚 Reference

This tool is based on:

> **Rockwell, B.W., 2021**, Mapping acid-generating minerals, acidic drainage, iron sulfate minerals, and other mineral groups using Landsat 8 Operational Land Imager data, San Juan Mountains, Colorado, and Four Corners Region. *U.S. Geological Survey Scientific Investigations Map 3466*.

---

## 🛠️ Troubleshooting

**Q: Authentication failed?**
- Make sure you have a GEE account: https://earthengine.google.com/
- Try running `ee.Authenticate()` manually

**Q: No data found?**
- Check your date range - summer months work best
- Reduce cloud cover threshold
- Verify your geometry is correct

**Q: Results look wrong?**
- Compare with true color imagery
- Check the classification legend
- Try adjusting thresholds

---

**Happy AMD Detection! 🔬**